# Actividad 6 — Implementación de un evaluador de cadenas para AFD y AFN

**Materia:** Lenguajes de Computación · Otoño 2026

| Nombre completo | No. de cuenta |
|---|---|
| Angel Rugerio Jiménez | 201720 |

## Objetivo

Implementar un evaluador de AFD y de AFN a través de un servicio web, donde **cada evaluador
cuenta con su propio endpoint**.

## Qué hace el servicio

El archivo [`main.py`](./main.py) levanta una aplicación de [FastAPI](https://fastapi.tiangolo.com/)
con dos endpoints de evaluación:

| Endpoint | Método | Autómata |
|---|---|---|
| `/afd/evaluar` | POST | Autómata finito **determinista** |
| `/afn/evaluar` | POST | Autómata finito **no determinista** (admite transiciones λ) |

Ambos reciben en el cuerpo de la petición:

* `tabla_transicion` — la tabla de transición del autómata,
* `estado_inicial`,
* `estados_finales`,
* `cadenas` — la(s) cadena(s) a evaluar,

y devuelven:

* **estados totales** (Q), **alfabeto** (Σ), **estado inicial** (q₀) y **estados finales** (F),
  todos deducidos de la propia tabla de transición;
* el **resultado de la evaluación de cada cadena**, incluyendo la **notación de transición**
  (la función δ paso a paso, la sucesión de configuraciones con `⊢`, la función extendida δ\* y
  el recorrido de estados).

La diferencia entre las dos tablas es la forma de cada celda:

$$\text{AFD: } \delta: Q \times \Sigma \to Q \qquad\qquad \text{AFN: } \delta: Q \times (\Sigma \cup \{\lambda\}) \to \mathcal{P}(Q)$$

es decir, en el AFD cada celda es **un** estado (`"q1"`) y en el AFN es un **conjunto** de estados
(`["q0", "q1"]`).

---
## 1. Preparación: levantar el servicio web

Se arranca `uvicorn` en un subproceso para que el notebook pueda hacer peticiones HTTP
**reales** a los endpoints (no se importan las funciones de `main.py` directamente: todos los
resultados de esta actividad se obtienen a través del servicio web).

In [1]:
import json
import subprocess
import sys
import time

import httpx

BASE = "http://127.0.0.1:8000"

servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Se espera a que el servidor responda antes de continuar
for _ in range(60):
    try:
        httpx.get(BASE + "/", timeout=1)
        break
    except Exception:
        time.sleep(0.5)

print("Servicio arriba:", httpx.get(BASE + "/").json()["actividad"])

Servicio arriba: Actividad 6 - Evaluador de cadenas para AFD y AFN


### Funciones auxiliares

`evaluar()` hace el POST al endpoint que corresponda y las funciones `mostrar_*` dan formato al
JSON de respuesta para poder leerlo en el notebook.

In [2]:
AUTOMATAS = json.load(open("automatas.json", encoding="utf-8"))


def evaluar(endpoint: str, automata: dict, cadenas=None) -> dict:
    """Envia el automata al endpoint indicado y regresa el JSON de respuesta."""
    cuerpo = {
        "estado_inicial": automata["estado_inicial"],
        "estados_finales": automata["estados_finales"],
        "tabla_transicion": automata["tabla_transicion"],
        "cadenas": automata["cadenas"] if cadenas is None else cadenas,
    }
    respuesta = httpx.post(BASE + endpoint, json=cuerpo, timeout=30)
    respuesta.raise_for_status()
    return respuesta.json()


def celda(valor) -> str:
    """Una celda de la tabla: un estado en el AFD, un conjunto en el AFN."""
    if valor is None:
        return "-"
    if isinstance(valor, str):
        return valor
    return "{" + ",".join(valor) + "}"


def mostrar_tabla(automata: dict, alfabeto=None) -> None:
    """Imprime la tabla de transicion con la notacion de clase (-> inicial, * final)."""
    tabla = automata["tabla_transicion"]
    if alfabeto is None:
        alfabeto = sorted({s for fila in tabla for s in tabla[fila]})
    ancho = max(8, max(len(q) for q in tabla) + 4)

    print(f"{'δ':<{ancho}}" + "".join(f"{s:<12}" for s in alfabeto))
    print("-" * (ancho + 12 * len(alfabeto)))
    for q in sorted(tabla):
        marca = ("->" if q == automata["estado_inicial"] else "  ") + \
                ("*" if q in automata["estados_finales"] else " ")
        print(f"{marca + q:<{ancho}}" + "".join(f"{celda(tabla[q].get(s)):<12}" for s in alfabeto))


def mostrar_definicion(r: dict) -> None:
    """Imprime la quintupla M = (Q, Σ, δ, q0, F) que devolvio el endpoint."""
    print(f"Tipo de automata : {r['tipo']}   {r['definicion_formal']}")
    print(f"Q  (estados)     : {{{', '.join(r['estados_totales'])}}}   -> {r['numero_de_estados']} estados")
    print(f"Σ  (alfabeto)    : {{{', '.join(r['alfabeto'])}}}")
    print(f"q0 (inicial)     : {r['estado_inicial']}")
    print(f"F  (finales)     : {{{', '.join(r['estados_finales'])}}}")
    if "tiene_transiciones_lambda" in r:
        print(f"Transiciones λ   : {'si' if r['tiene_transiciones_lambda'] else 'no'}")


def mostrar_resultados(r: dict) -> None:
    """Tabla resumen: una linea por cadena evaluada, con su notacion de transicion."""
    ancho = max(8, max(len(x["cadena"]) for x in r["resultados"]) + 2)
    print(f"{'CADENA':<{ancho}}{'RESULTADO':<14}NOTACION DE TRANSICION (funcion extendida)")
    print("-" * (ancho + 14 + 48))
    for x in r["resultados"]:
        veredicto = "ACEPTADA" if x["aceptada"] else "rechazada"
        print(f"{x['cadena']:<{ancho}}{veredicto:<14}{x['notacion_transicion']['funcion_extendida']}")
    print()
    print(f"Aceptadas  ({len(r['resumen']['aceptadas'])}): {r['resumen']['aceptadas']}")
    print(f"Rechazadas ({len(r['resumen']['rechazadas'])}): {r['resumen']['rechazadas']}")


def mostrar_traza(r: dict, cadena: str) -> None:
    """Detalle completo de la evaluacion de una sola cadena."""
    x = next(res for res in r["resultados"] if res["cadena"] == (cadena or "λ"))
    n = x["notacion_transicion"]
    print(f"Cadena: '{x['cadena']}'  ->  {'ACEPTADA' if x['aceptada'] else 'RECHAZADA'}")
    print()
    if "cerradura_inicial" in n:
        print("  Cerradura inicial:")
        print("    " + n["cerradura_inicial"])
    print("  Funcion de transicion paso a paso:")
    for paso in n["pasos"]:
        print("    " + paso)
    print()
    print("  Configuraciones:  " + n["configuraciones"])
    print("  Funcion extendida:  " + n["funcion_extendida"])
    if n.get("recorrido"):
        print("  Recorrido:  " + n["recorrido"])
    if n.get("camino_de_aceptacion"):
        print("  Camino de aceptacion:  " + n["camino_de_aceptacion"])
    print()
    print("  Motivo: " + x["motivo"])

---
## 2. El servicio y su documentación

`GET /` describe el servicio; FastAPI genera además la documentación interactiva en `/docs`.

In [3]:
print(json.dumps(httpx.get(BASE + "/").json(), indent=2, ensure_ascii=False))

{
  "actividad": "Actividad 6 - Evaluador de cadenas para AFD y AFN",
  "autor": "Angel Rugerio Jimenez - 201720",
  "endpoints": {
    "POST /afd/evaluar": "Evalua cadenas sobre un AFD",
    "POST /afn/evaluar": "Evalua cadenas sobre un AFN (con o sin transiciones λ)"
  },
  "documentacion_interactiva": "/docs"
}


---
## 3. Ejercicio 1 — Evaluador de **AFD**

Se usa el AFD construido en JFLAP para el **Reporte 1**, sobre el alfabeto
$\Sigma = \{a, b, c, d, e\}$ y con 10 estados. Es un AFD **completo**: existe exactamente una
transición para cada par (estado, símbolo). El estado $q_9$ es el **estado trampa**: es
absorbente y no es final, de modo que toda cadena que contenga el símbolo `e` termina rechazada.

Las 15 cadenas evaluadas son las del archivo de instrucciones del reporte.

In [4]:
afd = AUTOMATAS["ejercicio_1_afd"]
print(afd["nombre"], "\n")
print(afd["descripcion"], "\n")
mostrar_tabla(afd)

AFD del Reporte 1 (Σ = {a, b, c, d, e}, 10 estados) 

AFD completo trabajado en JFLAP para el Reporte 1. q9 es el estado trampa (absorbente y no final): cualquier cadena que contenga el simbolo 'e' es rechazada. 

δ       a           b           c           d           e           
--------------------------------------------------------------------
-> q0   q1          q2          q3          q5          q9          
   q1   q1          q2          q4          q5          q9          
  *q2   q4          q2          q6          q5          q9          
   q3   q1          q6          q3          q7          q9          
  *q4   q4          q2          q7          q8          q9          
   q5   q6          q2          q3          q5          q9          
  *q6   q6          q7          q8          q5          q9          
  *q7   q4          q7          q8          q7          q9          
  *q8   q6          q2          q8          q8          q9          
   q9   q9          q9     

### 3.1 Petición al endpoint `POST /afd/evaluar`

Este es el cuerpo JSON que se envía (recortado a las primeras filas de la tabla para que se lea):

In [5]:
cuerpo = {
    "estado_inicial": afd["estado_inicial"],
    "estados_finales": afd["estados_finales"],
    "tabla_transicion": afd["tabla_transicion"],
    "cadenas": afd["cadenas"],
}
print(json.dumps(cuerpo, indent=2, ensure_ascii=False)[:700] + "\n   ... (resto de la tabla) ...")

{
  "estado_inicial": "q0",
  "estados_finales": [
    "q2",
    "q4",
    "q6",
    "q7",
    "q8"
  ],
  "tabla_transicion": {
    "q0": {
      "a": "q1",
      "b": "q2",
      "c": "q3",
      "d": "q5",
      "e": "q9"
    },
    "q1": {
      "a": "q1",
      "b": "q2",
      "c": "q4",
      "d": "q5",
      "e": "q9"
    },
    "q2": {
      "a": "q4",
      "b": "q2",
      "c": "q6",
      "d": "q5",
      "e": "q9"
    },
    "q3": {
      "a": "q1",
      "b": "q6",
      "c": "q3",
      "d": "q7",
      "e": "q9"
    },
    "q4": {
      "a": "q4",
      "b": "q2",
      "c": "q7",
      "d": "q8",
      "e": "q9"
    },
    "q5": {
      "a": "q6",
      "b": "q2",
      "c":
   ... (resto de la tabla) ...


In [6]:
r_afd = evaluar("/afd/evaluar", afd)
mostrar_definicion(r_afd)

Tipo de automata : AFD   M = (Q, Σ, δ, q0, F)
Q  (estados)     : {q0, q1, q2, q3, q4, q5, q6, q7, q8, q9}   -> 10 estados
Σ  (alfabeto)    : {a, b, c, d, e}
q0 (inicial)     : q0
F  (finales)     : {q2, q4, q6, q7, q8}


### 3.2 Resultados de la evaluación de cada cadena

In [7]:
mostrar_resultados(r_afd)

CADENA  RESULTADO     NOTACION DE TRANSICION (funcion extendida)
----------------------------------------------------------------------
aaaab   ACEPTADA      δ*(q0, aaaab) = q2
cac     ACEPTADA      δ*(q0, cac) = q4
b       ACEPTADA      δ*(q0, b) = q2
abccc   ACEPTADA      δ*(q0, abccc) = q8
ab      ACEPTADA      δ*(q0, ab) = q2
aac     ACEPTADA      δ*(q0, aac) = q4
cccc    rechazada     δ*(q0, cccc) = q3
e       rechazada     δ*(q0, e) = q9
aaccc   ACEPTADA      δ*(q0, aaccc) = q8
bbc     ACEPTADA      δ*(q0, bbc) = q6
d       rechazada     δ*(q0, d) = q5
cdddd   ACEPTADA      δ*(q0, cdddd) = q7
ad      rechazada     δ*(q0, ad) = q5
bbd     rechazada     δ*(q0, bbd) = q5
bbbbb   ACEPTADA      δ*(q0, bbbbb) = q2

Aceptadas  (10): ['aaaab', 'cac', 'b', 'abccc', 'ab', 'aac', 'aaccc', 'bbc', 'cdddd', 'bbbbb']
Rechazadas (5): ['cccc', 'e', 'd', 'ad', 'bbd']


### 3.3 Notación de transición de algunas cadenas

Se muestran tres casos representativos: una cadena aceptada, una rechazada por terminar en un
estado no final, y una rechazada por caer en el estado trampa $q_9$.

In [8]:
mostrar_traza(r_afd, "abccc")

Cadena: 'abccc'  ->  ACEPTADA

  Funcion de transicion paso a paso:
    δ(q0, a) = q1
    δ(q1, b) = q2
    δ(q2, c) = q6
    δ(q6, c) = q8
    δ(q8, c) = q8

  Configuraciones:  (q0, abccc) ⊢ (q1, bccc) ⊢ (q2, ccc) ⊢ (q6, cc) ⊢ (q8, c) ⊢ (q8, λ)
  Funcion extendida:  δ*(q0, abccc) = q8
  Recorrido:  q0 --a--> q1 --b--> q2 --c--> q6 --c--> q8 --c--> q8

  Motivo: La cadena se consumio por completo y termino en 'q8', que SI es un estado final.


In [9]:
mostrar_traza(r_afd, "cccc")

Cadena: 'cccc'  ->  RECHAZADA

  Funcion de transicion paso a paso:
    δ(q0, c) = q3
    δ(q3, c) = q3
    δ(q3, c) = q3
    δ(q3, c) = q3

  Configuraciones:  (q0, cccc) ⊢ (q3, ccc) ⊢ (q3, cc) ⊢ (q3, c) ⊢ (q3, λ)
  Funcion extendida:  δ*(q0, cccc) = q3
  Recorrido:  q0 --c--> q3 --c--> q3 --c--> q3 --c--> q3

  Motivo: La cadena se consumio por completo y termino en 'q3', que NO es un estado final.


In [10]:
mostrar_traza(r_afd, "e")

Cadena: 'e'  ->  RECHAZADA

  Funcion de transicion paso a paso:
    δ(q0, e) = q9

  Configuraciones:  (q0, e) ⊢ (q9, λ)
  Funcion extendida:  δ*(q0, e) = q9
  Recorrido:  q0 --e--> q9

  Motivo: La cadena se consumio por completo y termino en 'q9', que NO es un estado final.


---
## 4. Ejercicio 2 — Evaluador de **AFN**

Se usa el AFN de la **Actividad 5**, sobre $\Sigma = \{a, b, c\}$, que reconoce

$$L = \{w \in \{a,b,c\}^* : w \text{ termina en } abc \text{ o en } acb\}$$

El no determinismo está en $q_0$: al leer `a` el autómata puede quedarse en $q_0$ (seguir
consumiendo prefijo) **o** pasar a $q_1$ (apostar a que ahí empieza el sufijo). Nótese que la
tabla ya no tiene un estado por celda sino un **conjunto** de estados, y que hay celdas vacías
(`-`), algo imposible en un AFD completo.

In [11]:
afn = AUTOMATAS["ejercicio_2_afn"]
print(afn["nombre"], "\n")
print(afn["descripcion"], "\n")
mostrar_tabla(afn, alfabeto=["a", "b", "c"])

AFN de la Actividad 5 (Σ = {a, b, c}) 

AFN que reconoce el lenguaje L = {w ∈ {a,b,c}* : w termina en 'abc' o en 'acb'}. El no determinismo esta en q0, donde el simbolo 'a' lleva a {q0, q1}. 

δ       a           b           c           
--------------------------------------------
-> q0   {q0,q1}     {q0}        {q0}        
   q1   -           {q2}        {q3}        
   q2   -           -           {q4}        
   q3   -           {q5}        -           
  *q4   -           -           -           
  *q5   -           -           -           


In [12]:
r_afn = evaluar("/afn/evaluar", afn)
mostrar_definicion(r_afn)

Tipo de automata : AFN   M = (Q, Σ, δ, q0, F)  con  δ: Q × (Σ ∪ {λ}) → P(Q)
Q  (estados)     : {q0, q1, q2, q3, q4, q5}   -> 6 estados
Σ  (alfabeto)    : {a, b, c}
q0 (inicial)     : q0
F  (finales)     : {q4, q5}
Transiciones λ   : no


### 4.1 Resultados de la evaluación de cada cadena

La cadena vacía se representa como `""` en la petición y se muestra como `λ` en la respuesta.

In [13]:
mostrar_resultados(r_afn)

CADENA  RESULTADO     NOTACION DE TRANSICION (funcion extendida)
----------------------------------------------------------------------
abc     ACEPTADA      δ*({q0}, abc) = {q0, q4}
acb     ACEPTADA      δ*({q0}, acb) = {q0, q5}
aabc    ACEPTADA      δ*({q0}, aabc) = {q0, q4}
bcacb   ACEPTADA      δ*({q0}, bcacb) = {q0, q5}
abcb    rechazada     δ*({q0}, abcb) = {q0}
cba     rechazada     δ*({q0}, cba) = {q0, q1}
λ       rechazada     δ*({q0}, λ) = {q0}
a       rechazada     δ*({q0}, a) = {q0, q1}
abcabc  ACEPTADA      δ*({q0}, abcabc) = {q0, q4}
ccacb   ACEPTADA      δ*({q0}, ccacb) = {q0, q5}
ab      rechazada     δ*({q0}, ab) = {q0, q2}

Aceptadas  (6): ['abc', 'acb', 'aabc', 'bcacb', 'abcabc', 'ccacb']
Rechazadas (5): ['abcb', 'cba', 'λ', 'a', 'ab']


### 4.2 Notación de transición

El evaluador simula el AFN por **conjuntos de estados** (construcción de subconjuntos sobre la
marcha), así que cada paso tiene la forma $\delta(\{q_i, \dots\}, a) = \{q_j, \dots\}$. Cuando la
cadena es aceptada se busca además, con una búsqueda en anchura, **un camino concreto** que la
acepte: eso es lo que hace visible el no determinismo.

In [14]:
mostrar_traza(r_afn, "aabc")

Cadena: 'aabc'  ->  ACEPTADA

  Cerradura inicial:
    cerradura-λ({q0}) = {q0}
  Funcion de transicion paso a paso:
    δ({q0}, a) = {q0, q1}
    δ({q0, q1}, a) = {q0, q1}
    δ({q0, q1}, b) = {q0, q2}
    δ({q0, q2}, c) = {q0, q4}

  Configuraciones:  ({q0}, aabc) ⊢ ({q0, q1}, abc) ⊢ ({q0, q1}, bc) ⊢ ({q0, q2}, c) ⊢ ({q0, q4}, λ)
  Funcion extendida:  δ*({q0}, aabc) = {q0, q4}
  Camino de aceptacion:  q0 --a--> q0 --a--> q1 --b--> q2 --c--> q4

  Motivo: Al terminar la cadena el conjunto alcanzado es {q0, q4} y contiene el/los estado(s) final(es) {q4}, asi que existe al menos un camino de aceptacion.


En el paso 3 de la cadena `aabc` se ve claramente el no determinismo: desde $\{q_0, q_1\}$ el
símbolo `b` lleva simultáneamente a $q_0$ (rama que sigue leyendo prefijo) y a $q_2$ (rama que ya
va a mitad del sufijo `abc`).

Ahora un caso rechazado: `ab` alcanza $q_2$, que está *a un símbolo* de ser final, pero la cadena
se acaba antes.

In [15]:
mostrar_traza(r_afn, "ab")

Cadena: 'ab'  ->  RECHAZADA

  Cerradura inicial:
    cerradura-λ({q0}) = {q0}
  Funcion de transicion paso a paso:
    δ({q0}, a) = {q0, q1}
    δ({q0, q1}, b) = {q0, q2}

  Configuraciones:  ({q0}, ab) ⊢ ({q0, q1}, b) ⊢ ({q0, q2}, λ)
  Funcion extendida:  δ*({q0}, ab) = {q0, q2}

  Motivo: Al terminar la cadena el conjunto alcanzado es {q0, q2} y ninguno de esos estados es final.


---
## 5. Ejercicio 3 — Evaluador de **AFN con transiciones λ**

Para ejercitar la **cerradura-λ** del evaluador se usa un AFN-λ que reconoce el lenguaje
$a^{*}b^{*}c^{*}$. Las transiciones $\lambda$ (columna `λ` de la tabla) permiten pasar del bloque
de las `a` al de las `b` y al de las `c` **sin consumir ningún símbolo**.

In [16]:
afnl = AUTOMATAS["ejercicio_3_afn_lambda"]
print(afnl["nombre"], "\n")
print(afnl["descripcion"], "\n")
mostrar_tabla(afnl, alfabeto=["a", "b", "c", "λ"])

AFN-λ que reconoce a*b*c* (Σ = {a, b, c}) 

AFN con transiciones lambda: las llaves "λ" permiten cambiar de bloque sin consumir simbolos, por lo que se ejercita la cerradura-λ del evaluador. 

δ       a           b           c           λ           
--------------------------------------------------------
-> q0   {q0}        -           -           {q1}        
   q1   -           {q1}        -           {q2}        
  *q2   -           -           {q2}        -           


In [17]:
r_afnl = evaluar("/afn/evaluar", afnl)
mostrar_definicion(r_afnl)

Tipo de automata : AFN-λ   M = (Q, Σ, δ, q0, F)  con  δ: Q × (Σ ∪ {λ}) → P(Q)
Q  (estados)     : {q0, q1, q2}   -> 3 estados
Σ  (alfabeto)    : {a, b, c}
q0 (inicial)     : q0
F  (finales)     : {q2}
Transiciones λ   : si


Obsérvese que el endpoint reporta el tipo como **AFN-λ** y que `λ` **no** aparece en el alfabeto
Σ, como corresponde a la definición formal.

### 5.1 Resultados de la evaluación de cada cadena

In [18]:
mostrar_resultados(r_afnl)

CADENA  RESULTADO     NOTACION DE TRANSICION (funcion extendida)
----------------------------------------------------------------------
λ       ACEPTADA      δ*({q0}, λ) = {q0, q1, q2}
abc     ACEPTADA      δ*({q0}, abc) = {q2}
aabbcc  ACEPTADA      δ*({q0}, aabbcc) = {q2}
ac      ACEPTADA      δ*({q0}, ac) = {q2}
b       ACEPTADA      δ*({q0}, b) = {q1, q2}
ccc     ACEPTADA      δ*({q0}, ccc) = {q2}
ba      rechazada     δ*({q0}, ba) = {}
cba     rechazada     δ*({q0}, cba) = {}
aaa     ACEPTADA      δ*({q0}, aaa) = {q0, q1, q2}

Aceptadas  (7): ['λ', 'abc', 'aabbcc', 'ac', 'b', 'ccc', 'aaa']
Rechazadas (2): ['ba', 'cba']


### 5.2 Notación de transición con cerradura-λ

En la traza aparece primero la cerradura-λ del estado inicial y, en cada paso, la cerradura-λ
aplicada al conjunto alcanzado.

In [19]:
mostrar_traza(r_afnl, "")

Cadena: 'λ'  ->  ACEPTADA

  Cerradura inicial:
    cerradura-λ({q0}) = {q0, q1, q2}
  Funcion de transicion paso a paso:

  Configuraciones:  ({q0, q1, q2}, λ)
  Funcion extendida:  δ*({q0}, λ) = {q0, q1, q2}
  Camino de aceptacion:  q0 --λ--> q1 --λ--> q2

  Motivo: Al terminar la cadena el conjunto alcanzado es {q0, q1, q2} y contiene el/los estado(s) final(es) {q2}, asi que existe al menos un camino de aceptacion.


In [20]:
mostrar_traza(r_afnl, "ac")

Cadena: 'ac'  ->  ACEPTADA

  Cerradura inicial:
    cerradura-λ({q0}) = {q0, q1, q2}
  Funcion de transicion paso a paso:
    δ({q0, q1, q2}, a) = {q0}  →  cerradura-λ = {q0, q1, q2}
    δ({q0, q1, q2}, c) = {q2}

  Configuraciones:  ({q0, q1, q2}, ac) ⊢ ({q0, q1, q2}, c) ⊢ ({q2}, λ)
  Funcion extendida:  δ*({q0}, ac) = {q2}
  Camino de aceptacion:  q0 --a--> q0 --λ--> q1 --λ--> q2 --c--> q2

  Motivo: Al terminar la cadena el conjunto alcanzado es {q2} y contiene el/los estado(s) final(es) {q2}, asi que existe al menos un camino de aceptacion.


La cadena `ba` se rechaza porque los bloques van en orden: una vez que se cruzó la λ-transición a
$q_1$ ya no hay manera de volver a leer una `a`. El conjunto de estados alcanzables se queda vacío.

In [21]:
mostrar_traza(r_afnl, "ba")

Cadena: 'ba'  ->  RECHAZADA

  Cerradura inicial:
    cerradura-λ({q0}) = {q0, q1, q2}
  Funcion de transicion paso a paso:
    δ({q0, q1, q2}, b) = {q1}  →  cerradura-λ = {q1, q2}
    δ({q1, q2}, a) = {}

  Configuraciones:  ({q0, q1, q2}, ba) ⊢ ({q1, q2}, a) ⊢ ({}, λ)
  Funcion extendida:  δ*({q0}, ba) = {}

  Motivo: El conjunto de estados alcanzables quedo vacio: ninguna rama del AFN pudo seguir leyendo la cadena.


---
## 6. Casos borde y validación de la entrada

### 6.1 AFD con tabla de transición incompleta (estado trampa implícito)

Si la tabla no define $\delta(q, a)$ para algún par, el evaluador no falla: detiene el recorrido e
informa en qué símbolo se quedó, que es equivalente a caer en un estado trampa.

In [22]:
parcial = {
    "estado_inicial": "q0",
    "estados_finales": ["q2"],
    "tabla_transicion": {"q0": {"a": "q1"}, "q1": {"b": "q2"}, "q2": {}},
    "cadenas": ["ab", "aa", "abb"],
}
r_parcial = evaluar("/afd/evaluar", parcial)
mostrar_definicion(r_parcial)
print()
mostrar_resultados(r_parcial)
print()
mostrar_traza(r_parcial, "aa")

Tipo de automata : AFD   M = (Q, Σ, δ, q0, F)
Q  (estados)     : {q0, q1, q2}   -> 3 estados
Σ  (alfabeto)    : {a, b}
q0 (inicial)     : q0
F  (finales)     : {q2}

CADENA  RESULTADO     NOTACION DE TRANSICION (funcion extendida)
----------------------------------------------------------------------
ab      ACEPTADA      δ*(q0, ab) = q2
aa      rechazada     δ*(q0, aa) = indefinida
abb     rechazada     δ*(q0, abb) = indefinida

Aceptadas  (1): ['ab']
Rechazadas (2): ['aa', 'abb']

Cadena: 'aa'  ->  RECHAZADA

  Funcion de transicion paso a paso:
    δ(q0, a) = q1

  Configuraciones:  (q0, aa) ⊢ (q1, a)
  Funcion extendida:  δ*(q0, aa) = indefinida
  Recorrido:  q0 --a--> q1

  Motivo: No existe la transicion delta(q1, a); el AFD se detiene en el simbolo 2 de la cadena (la cadena cae en el estado trampa implicito).


### 6.2 Errores de definición del autómata

El servicio responde `400` si el estado inicial o alguno de los estados finales no aparece en la
tabla de transición (una defensa contra estados mal escritos).

In [23]:
malos = [
    ("estado inicial inexistente",
     {"estado_inicial": "qX", "estados_finales": ["q2"],
      "tabla_transicion": {"q0": {"a": "q1"}, "q1": {"b": "q2"}, "q2": {}}, "cadenas": ["ab"]}),
    ("estado final inexistente",
     {"estado_inicial": "q0", "estados_finales": ["q9"],
      "tabla_transicion": {"q0": {"a": "q1"}, "q1": {"b": "q2"}, "q2": {}}, "cadenas": ["ab"]}),
]
for descripcion, cuerpo_malo in malos:
    resp = httpx.post(BASE + "/afd/evaluar", json=cuerpo_malo, timeout=10)
    print(f"{descripcion:<32} -> HTTP {resp.status_code}: {resp.json()['detail']}")

estado inicial inexistente       -> HTTP 400: El estado inicial 'qX' no aparece en el automata.
estado final inexistente         -> HTTP 400: Estados finales que no aparecen en el automata: ['q9']


### 6.3 El endpoint de AFD rechaza tablas no deterministas

El tipo declarado del AFD es `Dict[str, Dict[str, str]]`, así que si una celda trae un conjunto de
estados Pydantic devuelve `422` y obliga a usar el endpoint de AFN.

In [24]:
no_determinista = {
    "estado_inicial": "q0", "estados_finales": ["q1"],
    "tabla_transicion": {"q0": {"a": ["q0", "q1"]}, "q1": {}}, "cadenas": ["a"],
}
resp = httpx.post(BASE + "/afd/evaluar", json=no_determinista, timeout=10)
print("POST /afd/evaluar ->", resp.status_code, resp.json()["detail"][0]["msg"])

resp = httpx.post(BASE + "/afn/evaluar", json=no_determinista, timeout=10)
print("POST /afn/evaluar ->", resp.status_code, "|",
      [(x["cadena"], x["aceptada"]) for x in resp.json()["resultados"]])

POST /afd/evaluar -> 422 Input should be a valid string
POST /afn/evaluar -> 200 | [('a', True)]


---
## 7. Conclusiones

* Los dos evaluadores comparten la lectura de la tabla de transición (de ahí salen $Q$ y $\Sigma$),
  pero difieren en la simulación: el AFD mantiene **un estado actual** y el AFN mantiene un
  **conjunto de estados actuales**, que es justo la construcción de subconjuntos aplicada sobre la
  marcha.
* La forma de la tabla basta para distinguir los dos autómatas: una celda con un estado (AFD)
  contra una celda con un conjunto de estados (AFN). Por eso el endpoint de AFD puede rechazar
  automáticamente una tabla no determinista (§6.3).
* Un AFD acepta una cadena si el **único** recorrido posible termina en un estado final; un AFN la
  acepta si **existe al menos un** recorrido que termine en un estado final, aunque otros mueran o
  terminen en estados no finales. Eso se ve en `aabc` (§4.2), donde la rama que se queda en $q_0$
  no acepta y la que salta a $q_1$ sí.
* Las transiciones $\lambda$ no agregan poder de reconocimiento, pero sí obligan a calcular la
  **cerradura-λ** antes y después de cada símbolo; sin ella la cadena vacía sería rechazada por el
  autómata del ejercicio 3, que sí la acepta.

In [25]:
servidor.terminate()
servidor.wait(timeout=10)
print("Servicio detenido.")

Servicio detenido.